# B3I — Masked-input SAM2 (test 1)

## Why
B3S and B3U showed that surplus masks are easier to **prevent than to remove**: post-hoc filters always cost real objects in dense scenes. OmniCount feeds SAM an RGB patch I·S (image × refined semantic mask), so background cannot be segmented at all. We never tested this: SAM2 always saw the full image.

## Pre-declared design (agreed before any B3I result)
| Choice | Rule |
|---|---|
| Semantic region | pixels where the **target wins** in the tiled expanded-vocabulary map (B3S Step 5 cache) |
| Variants | **M0:** region as is; **M1:** region dilated by r = round(0.02 × short side) px (D1's existing peak spacing). Opening is not used (Group B: it removes small objects) |
| Masked image | RGB × region (outside pixels set to 0, black) |
| SAM2 | SAM2.1-Small (frozen), unchanged: default automatic mask generator (B0) and point prompting (multimask off) |
| Prompts | the cached `dense_recovery` prompts (full-image evidence) — only SAM2's input changes |
| Discard rule | drop any mask with more than half its pixels outside the (variant's) region |
| Counting | unchanged: GPV_R4 (NMS 0.5 + B0 pool), then H1b on those masks (A3c, depth, D1 from the full image) |
| Reported | B0-masked, GPV_R4-masked, H1b-masked for M0 and M1 |
| Targets | T1 (S1, S2 not sig. worse than B0_v1); T2′ (S5, S6 not sig. worse than H1b_v1); T3′ (reweighted MAE sig. better than H1b_v1) |

Test 2 (SAM2.1-Large, full image) runs in parallel in `B3L_SAM2_Large.ipynb`.

## Run order
T4 → 1.0, 1.1, 1.2 → 2.1, 2.2 → 3.1 (caching, ~75–85 min, resumable) → 4.1. Step 4.1 can also run later on CPU after 1.0–1.2.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3I_Masked_Input_SAM2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3I_Masked_Input_SAM2


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


# 2. SAM2.1-Small

In [4]:
# ============================================================
# Step 2.1 — Install SAM2 (frozen commit)
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
print("SAM2 commit:", subprocess.check_output(["git", "-C", "/content/sam2_repo", "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.1 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [5]:
# ============================================================
# Step 2.2 — Build the frozen SAM2.1-Small AMG and predictor
# ============================================================

SAM2_S = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"
amg = b0.build_amg(SAM2_S, device=DEVICE)                     # frozen B0 setup (SHA verified)
predictor = SAM2ImagePredictor(build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_S), device=DEVICE).eval())
print("SAM2.1-Small AMG and predictor ready")

SAM2.1-Small AMG and predictor ready


# 3. Masked-input caching (M0, M1)

In [6]:
# ============================================================
# Step 3.1 — Masked-input B0 and guided masks on train_dev400 (resumable)
# ============================================================

from scipy import ndimage
TD_TILED = TD_ROOT / "san_vocab_tiled"
VARIANTS = ["M0", "M1"]
MI_ROOT = TD_ROOT / "masked_input"
for v in VARIANTS:
    (MI_ROOT / v / "b0").mkdir(parents=True, exist_ok=True)
    (MI_ROOT / v / "guided").mkdir(parents=True, exist_ok=True)
(MI_ROOT / "_done").mkdir(parents=True, exist_ok=True)

def region_for(variant, amax):
    reg = amax == 0
    if variant == "M1" and reg.any():
        r = int(round(0.02 * min(reg.shape)))
        reg = ndimage.distance_transform_edt(~reg) <= r
    return reg

def keep_inside(masks, region):
    if masks.shape[0] == 0:
        return np.zeros(0, bool)
    inside = (masks & region[None]).reshape(masks.shape[0], -1).sum(1)
    total = masks.reshape(masks.shape[0], -1).sum(1)
    return inside > 0.5 * np.maximum(total, 1)

todo = [r for r in td_df.itertuples(index=False)
        if not (MI_ROOT / "_done" / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for r in tqdm(todo, desc="Masked-input SAM2"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)              # as frozen B0
    with np.load(TD_TILED / f"{stem}.npz") as v:
        amax = v["tiled_argmax"]
    prompts = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")["prompts_xy"]
    info = {"dataset_index": int(di)}
    for var in VARIANTS:
        reg = region_for(var, amax)
        masked = (img * reg[..., None]).astype(np.uint8)
        bm, bmeta = b0.run_amg(amg, masked)
        kb = keep_inside(bm, reg)
        io_cache.save_b0_masks(MI_ROOT / var / "b0" / f"{int(di):05d}_{stem}.npz", bm[kb], filter_meta(bmeta, kb))
        gm, gs = guided_masks_for_prompts(predictor, masked, prompts)
        kg = keep_inside(gm, reg)
        io_cache.save_guided_masks(MI_ROOT / var / "guided" / f"{int(di):04d}_{stem}.npz",
                                   gm[kg], prompts[kg] if len(prompts) else prompts, gs[kg],
                                   img.shape[:2], dataset_index=di, image_id=r.image_id)
        info.update({f"{var}_region_frac": float(reg.mean()), f"{var}_b0_raw": int(bm.shape[0]),
                     f"{var}_b0_kept": int(kb.sum()), f"{var}_guided_kept": int(kg.sum())})
    with open(MI_ROOT / "_done" / f"{int(di):05d}.json", "w") as f:
        json.dump(info, f)
    if (todo.index(r) + 1) % 50 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


Masked-input SAM2:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 87.3 min


# 4. Systems and targets

In [7]:
# ============================================================
# Step 4.1 — B0 / GPV_R4 / H1b on masked input; targets (CPU-capable)
# ============================================================

assert "systems_from_masks" in globals(), "Run Steps 1.0-1.2 first."
MI_ROOT = TD_ROOT / "masked_input"; VARIANTS = ["M0", "M1"]
rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Systems"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    info = json.load(open(MI_ROOT / "_done" / f"{int(di):05d}.json"))
    row = {"dataset_index": di, **info}
    for var in VARIANTS:
        bm, bmeta = io_cache.load_b0_masks(MI_ROOT / var / "b0" / f"{int(di):05d}_{stem}.npz")
        g = io_cache.load_guided_masks(MI_ROOT / var / "guided" / f"{int(di):04d}_{stem}.npz")
        s = systems_from_masks(bm, bmeta["predicted_iou"], g["masks"], g["scores"], a3c, dep, san["target_score"])
        row.update({f"{k}[{var}]": v for k, v in s.items()})
    rows.append(row)
mi = pd.DataFrame(rows).merge(
    td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
    .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one")

print("Semantic region size and masks kept, by stratum (means):")
display(mi.groupby("density_stratum")[[c for c in mi.columns if c.endswith(("region_frac", "_b0_raw", "_b0_kept", "_guided_kept"))]]
        .mean().reindex(STRATA).round(2))
print("Images with an empty semantic region (M0):", int((mi["M0_region_frac"] == 0).sum()))

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1"] + [f"{k}[{v}]" for v in VARIANTS for k in ["B0", "GPV_R4", "H1b"]]
res, targets = evaluate(mi, SYS, "Masked-input SAM2.1-Small (train_dev400)")

mi.to_csv(OUT_DIR / "B3I_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3I_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3I_targets.csv", index=False)
with open(OUT_DIR / "B3I_metadata.json", "w") as f:
    json.dump({"notebook": "B3I_Masked_Input_SAM2", "test": "1 — masked input, SAM2.1-Small",
               "region": "target-winning pixels, tiled expanded-vocabulary map",
               "variants": {"M0": "no dilation", "M1": "dilated by round(0.02 x short side)"},
               "discard": "mask dropped if > 50% of its pixels lie outside the region",
               "prompts": "cached dense_recovery (full-image evidence)",
               "counting": "GPV_R4 (NMS 0.5 + B0 pool), H1b on those masks",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Systems:   0%|          | 0/400 [00:00<?, ?it/s]

Semantic region size and masks kept, by stratum (means):


,M0_region_frac,M0_b0_raw,M0_b0_kept,M0_guided_kept,M1_region_frac,M1_b0_raw,M1_b0_kept,M1_guided_kept
density_stratum,,,,,,,,
S1_0_20,0.36,11.84,10.55,30.05,0.44,11.01,9.80,30.55
S2_20_40,0.43,15.95,14.56,33.95,0.50,15.40,14.02,34.61
S3_40_60,0.54,24.27,22.32,56.55,0.61,22.62,21.47,58.10
S4_60_80,0.53,31.43,30.30,50.53,0.60,30.37,29.35,51.73
S5_80_95,0.57,52.60,51.07,61.93,0.65,52.70,51.37,63.70
S6_95_100,0.64,59.45,58.23,80.62,0.71,61.80,60.97,83.73


Images with an empty semantic region (M0): 5
Masked-input SAM2.1-Small (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,B0[M0],36.44,-65.73,4.97,6.84,11.22,24.47,63.27,349.02
4,GPV_R4[M0],30.98,-53.47,6.11,6.01,11.55,16.43,47.75,315.92
5,H1b[M0],29.58,-49.12,7.41,6.89,12.05,14.83,42.33,299.88
6,B0[M1],36.38,-65.81,4.62,6.88,11.73,25.15,62.57,346.28
7,GPV_R4[M1],31.09,-51.76,7.04,6.90,11.80,17.25,46.02,311.85
8,H1b[M1],29.70,-47.76,7.99,7.81,11.80,15.33,41.80,296.78


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,B0[M0],"-0.59 [-1.84, +0.47]","-0.90 [-2.22, +0.33]","+25.93 [+17.88, +34.98]","+74.03 [+60.63, +88.15]","+7.79 [+5.92, +9.70]",True,False,False,False
1,GPV_R4[M0],"+0.55 [-1.10, +2.15]","-1.72 [-3.33, -0.03]","+10.42 [+4.18, +18.38]","+40.93 [+28.70, +54.60]","+2.33 [+0.87, +3.89]",True,False,False,False
2,H1b[M0],"+1.85 [-0.08, +3.78]","-0.85 [-2.77, +1.12]","+5.00 [-0.07, +11.45]","+24.90 [+13.83, +37.88]","+0.93 [-0.33, +2.27]",True,False,False,False
3,B0[M1],"-0.94 [-2.11, +0.12]","-0.86 [-2.11, +0.33]","+25.23 [+17.12, +34.40]","+71.30 [+59.08, +84.67]","+7.73 [+5.84, +9.62]",True,False,False,False
4,GPV_R4[M1],"+1.47 [-0.04, +2.94]","-0.84 [-2.44, +0.83]","+8.68 [+2.87, +16.37]","+36.87 [+24.97, +50.05]","+2.44 [+1.05, +3.95]",True,False,False,False
5,H1b[M1],"+2.42 [+0.67, +4.11]","+0.08 [-1.76, +1.99]","+4.47 [+0.47, +10.45]","+21.80 [+11.07, +34.38]","+1.05 [-0.05, +2.28]",False,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3I_Masked_Input_SAM2
